# Topic 1 — Auditability and calibration of LLM-based satisfaction scoring

**Preliminary research notebook (protocol).** This is a pilot, not the paper. All data are
synthetic; no claim here is about real callers, real agents, or frontier LLMs.

The object of study is a call-eval-style measurement chain:

```
transcript -> cited evidence spans -> typed facts -> deterministic score -> survey calibration
```

The question is whether that chain stays **calibrated** and **auditable** when the synthetic
data-generating process shifts along Mirage-style dimensions (disfluency, emotional arcs,
persona mix, lexical paraphrase).

**Status.** The offline extractor is a strict lexicon baseline; a small open model (Qwen2.5-3B,
4-bit, local MLX) is used only as a feasibility pilot for direct scoring. The results are a
stress test of the *architecture and audit metrics*, not a benchmark of LLM quality.

## Research question

> When satisfaction scores are produced through an evidence-chain plus survey-calibration
> pipeline on synthetic call logs, do calibration and evidence consistency survive
> distributional shift — and can runtime audit signals flag the calls that break?

### Hypotheses and pre-specified readings

| ID | Hypothesis | Operationalisation | Kept if |
| --- | --- | --- | --- |
| **H1** | Calibration degrades under shift. | Holdout ECE and calibration slope after applying source-fitted calibration to each shift, vs source holdout. | ΔECE > 0.03 with bootstrap CI excluding 0, or slope moves away from 1 by > 0.15. |
| **H2** | Failures localise before calibration, at the span/fact layer. | Gold evidence-turn hit rate, fact accuracy, and paired score-change rate on label-preserving shifts. | Evidence hit or fact accuracy drops > 5 pp while latent labels are unchanged. |
| **H3** | Runtime evidence flags predict large scoring errors better than model confidence. | AUC of missing-evidence / conflict flags vs AUC of posterior confidence for `abs(error) > 1`. | Audit AUC exceeds confidence AUC by > 0.05. |
| **H4** | Conformal coverage is not robust to synthetic shift. | 90% split-conformal sets fitted on source, evaluated on each target. | Coverage falls below 0.85 under at least one shift. |

H3 and H4 are deliberately falsifiable; a negative result is a finding, not a failure.

**Power note.** With 100 target calls, ECE bootstrap intervals are wide enough to overlap
across source and the mild shifts. H1 is therefore only claimed where the paired ΔQWK interval
also excludes zero; otherwise the result is reported as directional.

## Design

| Component | Source | Shifted targets |
| --- | --- | --- |
| Calls | 240 train + 80 dev + 100 sealed holdout | 100 paired calls per shift |
| Labels | latent 1–5, noisy survey (~72% response), two simulated annotators | latent/survey held fixed for paired shifts |
| Calibration | fit on source train+dev surveys only | applied unchanged to targets |
| Shifts | — | disfluency (label-preserving), emotional arc (concept), paraphrase (lexical), persona mix (covariate), combined |
| Extractor | strict offline lexicon | same |
| Scores | 1–5 deterministic fact score; ordinal + isotonic calibration | same |

The **persona-mix** target is unpaired (new calls, different persona weights); all other
shifts are paired with the source holdout, which enables span/fact/score consistency checks.

## Keep/discard loop

Every code change is an experiment card with a pre-specified decision rule:

1. State the change and the metric it should move.
2. Run it on source **and** shifted targets.
3. Read the result table.
4. Record **KEEP** or **DISCARD** in `03_iterations.ipynb` and `EXPERIMENT_LOG.md`.

Thresholds: keep a change if it improves the target metric by at least the pre-specified
margin without a source regression larger than 0.02 QWK. Changes that only help because they
encode knowledge of the perturbation (e.g. a lexicon written from the paraphrase list) are
marked **KEEP (diagnostic, circular)** and are not treated as deployable fixes.

## What this pilot cannot show

- It cannot rank LLMs: the LLM arm is one 3B open model on 40 calls per condition.
- It cannot validate the synthetic labels: the “ground truth” is the generator.
- It cannot establish real-world calibration: surveys are simulated and noisy.
- It cannot claim the offline lexicon represents an LLM extractor; it is a transparent
  lower bound / stress-test instrument.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from callsat_audit.config import FIGURES_DIR, ROOT
from callsat_audit.generate import load_datasets, summary_table

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

datasets = load_datasets()
print(summary_table(datasets))

combined       n=100 survey= 73 latent_mean=3.55
disfluency     n=100 survey= 73 latent_mean=3.55
emotional_arc  n=100 survey= 73 latent_mean=3.55
paraphrase     n=100 survey= 73 latent_mean=3.55
persona_mix    n=100 survey= 74 latent_mean=3.11
source         n=420 survey=300 latent_mean=3.37


In [2]:
rows = []
for condition, calls in datasets.items():
    labels = np.array([c.latent_satisfaction for c in calls])
    surveys = np.array([c.survey_response for c in calls if c.survey_response is not None])
    rows.append(
        {
            "condition": condition,
            "n": len(calls),
            "survey_rate": len(surveys) / len(calls),
            "latent_mean": labels.mean(),
            "latent_sd": labels.std(),
            "pct_resolved": np.mean([c.gold_resolution == "resolved" for c in calls]),
            "mean_effort": np.mean([c.gold_effort for c in calls]),
            "mean_sentiment": np.mean([c.gold_sentiment for c in calls]),
        }
    )
pd.DataFrame(rows).set_index("condition").round(3)

,n,survey_rate,latent_mean,latent_sd,pct_resolved,mean_effort,mean_sentiment
condition,,,,,,,
combined,100,0.730,3.550,1.152,0.66,1.110,-0.540
disfluency,100,0.730,3.550,1.152,0.66,1.110,0.130
emotional_arc,100,0.730,3.550,1.152,0.66,1.110,-0.700
paraphrase,100,0.730,3.550,1.152,0.66,1.110,0.130
persona_mix,100,0.740,3.110,1.207,0.61,1.230,-0.080
source,420,0.714,3.371,1.203,0.61,1.195,0.117


In [3]:
# Invariant check: paired shifts must not change the latent label or survey.
source_holdout = {c.call_id: c for c in datasets["source"] if c.split == "holdout"}
checks = []
for condition in ("disfluency", "emotional_arc", "paraphrase", "combined"):
    paired = [c for c in datasets[condition] if c.call_id in source_holdout]
    label_changes = sum(
        c.latent_satisfaction != source_holdout[c.call_id].latent_satisfaction for c in paired
    )
    survey_changes = sum(
        c.survey_response != source_holdout[c.call_id].survey_response for c in paired
    )
    checks.append(
        {
            "condition": condition,
            "paired": len(paired),
            "latent_changes": label_changes,
            "survey_changes": survey_changes,
        }
    )
pd.DataFrame(checks).set_index("condition")

,paired,latent_changes,survey_changes
condition,,,
disfluency,100,0,0
emotional_arc,100,0,0
paraphrase,100,0,0
combined,100,0,0


## A source call and its gold evidence

The generator records every cue span, so the audit can distinguish three failure loci:
span missed → fact wrong → score/calibration shift. The example below shows the rendered
transcript plus the gold facts that a perfect extractor would recover.

In [4]:
example = next(
    c for c in datasets["source"] if c.split == "holdout" and c.meta.get("implicit_resolution")
)
print(example.transcript())
print()
print(
    {
        "latent": example.latent_satisfaction,
        "survey": example.survey_response,
        "gold": {
            k: example.gold_fact(k)
            for k in ("resolved", "effort", "sentiment", "empathy")
        },
        "meta": example.meta,
    }
)

[1] AGENT: Thanks for calling Meridian Support. This is Marcus. How can I help today?
[2] CUSTOMER: I'm still waiting on a refund I asked for two weeks ago. I've got about five minutes before my next meeting.
[3] AGENT: Could you verify the postcode on the account?
[4] CUSTOMER: It was $120.50, returned on the Wednesday.
[5] AGENT: Okay, I've found the record. Let me take a look.
[6] AGENT: I'll need to put you on a brief hold while I check that.
[7] AGENT: I'll have to transfer you to our billing team for this.
[8] AGENT: You'll need to call back tomorrow with the reference number.
[9] AGENT: Unfortunately, that's not something we can do.
[10] CUSTOMER: I'm not really happy with that.
[11] AGENT: Anything else I can do for you?
[12] CUSTOMER: That's everything. Thank you.

{'latent': 2, 'survey': 1, 'gold': {'resolved': 'unresolved', 'effort': 3, 'sentiment': -1, 'empathy': 0}, 'meta': {'shift': 'none', 'agent_style': 'transactional', 'implicit_resolution': True}}
